# Bridge A+B Comparison Analysis\n\nLoad an existing `experiment=comparison train=ab_only data=bridge_ot|bridge_random` run and inspect baseline vs constrained metrics and rollout plots.

In [ ]:
from pathlib import Path\nimport json\n\nfrom IPython.display import Image, display\n

In [ ]:
def find_project_root(start: Path) -> Path:\n    cur = start.resolve()\n    for candidate in [cur, *cur.parents]:\n        if (candidate / 'configs').exists() and (candidate / 'src').exists():\n            return candidate\n    raise RuntimeError('Could not locate project root containing configs/ and src/.')\n\nROOT = find_project_root(Path.cwd())\nOUTPUTS = ROOT / 'outputs'\n\n# Option 1: set RUN_DIR manually to a comparison run folder.\n# RUN_DIR = ROOT / 'outputs/2026-04-27/bridge_ab_ot_ab_only/18-30-00'\n\n# Option 2: auto-pick latest bridge_ab_ot_ab_only run.\ncandidates = sorted(OUTPUTS.glob('*/bridge_ab_ot_ab_only/*'))\nassert candidates, 'No run found for pattern outputs/*/bridge_ab_ot_ab_only/*'\nRUN_DIR = candidates[-1]\n\nprint('Using RUN_DIR:', RUN_DIR)

In [ ]:
comparison_path = RUN_DIR / 'comparison.json'\nassert comparison_path.exists(), f'Missing comparison.json: {comparison_path}'\ncomparison = json.loads(comparison_path.read_text())\n\nbaseline = comparison['baseline']\nconstrained = comparison['constrained']\n\nkeys = [\n    'constraint_residual_avg',\n    'intermediate_empirical_w2_avg',\n    'transport_endpoint_empirical_w2',\n]\n\nprint('meta:', comparison.get('meta', {}))\nprint('')\nprint(f"{'metric':36s} {'baseline':>14s} {'constrained':>14s} {'delta(c-b)':>14s}")\nprint('-' * 82)\nfor key in keys:\n    b = baseline.get(key)\n    c = constrained.get(key)\n    if b is None or c is None:\n        print(f"{key:36s} {str(b):>14s} {str(c):>14s} {'n/a':>14s}")\n    else:\n        delta = float(c) - float(b)\n        print(f"{key:36s} {float(b):14.6f} {float(c):14.6f} {delta:14.6f}")

In [ ]:
for mode in ['baseline', 'constrained']:\n    mode_dir = RUN_DIR / mode\n    print('\n' + '=' * 18 + f' {mode.upper()} ' + '=' * 18)\n    for name in ['rollout_marginal_grid.png', 'rollout_empirical_w2.png', 'sample_paths.png', 'constraint_residuals.png']:\n        p = mode_dir / name\n        if p.exists():\n            print(name)\n            display(Image(filename=str(p)))\n        else:\n            print(f'Missing: {p}')